<img src="images/header.png" width=180, align="center"/>

Master's degree in Intelligent Systems

Subject: 11754 - Deep Learning

Year: 2025-2026

Professor: Miguel Ángel Calafat Torrens

# LAB 3 — Generative Adversarial Networks

**In this lab you have to deliver only this file.**

This lab builds on the concepts from LESSON 3A (classical GANs, mode collapse) and LESSON 3B (WGAN-GP, Wasserstein distance, latent space). You will complete 5 exercises that require both implementation and written analysis.

## Exercises Overview
- **Exercise A** — Comparative Analysis: Classical GAN vs WGAN-GP on CelebA (medium GPU)
- **Exercise B** — Latent Space Interpolation (MNIST + CelebA, low GPU)
- **Exercise C** — Hyperparameter Sensitivity: gamma values (MNIST, low GPU)
- **Exercise D** — Mode Collapse + Diversity Metric (MNIST, low GPU)
- **Exercise E** — Critical Thinking (written, no GPU)

**You can modify, add or remove any cell that you want to fulfill the requirements.**

**Written analysis guidelines:** All exercises with written components should demonstrate understanding of the underlying concepts. Exercise A requires a minimum of 200 words; for other exercises, aim for clear, substantive answers (typically 50-150 words each).

In [ ]:
# Connect to your drive
from google.colab import drive, files
drive.mount('/content/gdrive')
%cd '/content/gdrive/MyDrive/LABS2026/LAB03'
%ls -l

import pathlib
import sys
import os
import helper_L3 as hp

PROJECT_DIR = str(pathlib.Path().resolve())
sys.path.append(PROJECT_DIR)

In [ ]:
# Imports
import PIL
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# --- CelebA Dataset Setup ---
dataset_zip_fullpath = '/content/gdrive/MyDrive/datasets/img_align_celeba_small_10000.zip'
DATA_DIR = hp.extract_dataset(dataset_zip_fullpath, remove_zip=True)

img_size = 64
transform = transforms.Compose([
    transforms.CenterCrop((178, 178)),
    transforms.Resize((img_size, img_size)),
    hp.ToScaledTensor(),
])

celeba_dataset = hp.CustomDataset(DATA_DIR, transforms=transform, lim=-1)
celeba_loader = DataLoader(celeba_dataset, batch_size=128, shuffle=True)

In [ ]:
# --- MNIST Dataset Setup ---
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
mnist_dataset = datasets.MNIST(root='data', train=True, download=True,
                               transform=mnist_transform)
mnist_loader = DataLoader(mnist_dataset, batch_size=128, shuffle=True)

---
# Exercise A — Comparative Analysis (CelebA, medium GPU)

**Requirements:**
1. Define a **Critic** and a **Generator** with **5+ convolutional layers** and **z_dim=200**.
   (Hint: `z_dim` is a constructor parameter — you only need to pass it when instantiating
   the network, no structural changes are needed.)
2. Define a **Discriminator** with the same architecture as the Critic but using **BatchNorm**
   instead of InstanceNorm (following DCGAN conventions from LESSON 3A).
3. Train a **classical GAN** for **40+ epochs** using `hp.gan_loss_fcn`.
   (Hint: use the same hyperparameters as in LESSON 3A for the classical GAN.)
4. Train a **WGAN-GP** for **40+ epochs** using `hp.Wasserstein_loss_fcn` + `hp.penalty_fcn`.
   (Hint: use `crit_cycles=2` — this balances critic and generator training for this
   dataset size. A learning rate around `1e-4` works well for 5+ block architectures.)
5. Save checkpoints at **epochs 20 and 40** for both trainings.
   (Hint: set `save_step=20` in your config dict. Use `checkpoint_prefix` to avoid
   filename collisions between different training runs, e.g., `'checkpoint_prefix': 'classical'`.)
6. Present:
   - Loss curves for both (side by side)
   - Generated images at epochs 20 and 40 (for both)
   - **Written analysis** comparing training stability, image quality,
     and convergence behavior

In [ ]:
# Exercise A — Define your Critic (5+ conv blocks, z_dim=200)

In [ ]:
# Exercise A — Define your Generator (5+ conv blocks, z_dim=200)

In [ ]:
# Exercise A — Define your Discriminator (same architecture as Critic but with BatchNorm)

In [ ]:
# Exercise A — Train classical GAN (40+ epochs)

In [ ]:
# Exercise A — Train WGAN-GP (40+ epochs)

In [ ]:
# Exercise A — Load checkpoints, show images, plot loss curves

**Exercise A — Written Analysis (minimum 200 words):**

*Write your analysis here comparing classical GAN vs WGAN-GP in terms of training stability, image quality, and convergence behavior.*

---
# Exercise B — Latent Space Interpolation (MNIST + CelebA, low GPU)

**Requirements:**

**Part 1 — MNIST interpolation:**
1. Train an FC-GAN on MNIST (~20 epochs) using the architecture from LESSON 3A.
2. Take 2 random latent vectors.
3. Generate **12 images** by interpolating between the two vectors: the two endpoints and
   10 evenly-spaced intermediate points, using `torch.linspace(0, 1, 12)`.
4. Display all 12 images as a single row.

**Part 2 — CelebA interpolation:**
5. Using the trained WGAN-GP generator from Exercise A (z_dim=200).
6. Repeat the same interpolation procedure (2 latent vectors, 12 images, single row).

**Written analysis:**
7. **Compare** the two interpolations. What differences do you observe in the smoothness
   and visibility of transitions? Why does MNIST produce more dramatic changes than CelebA?
   What does this tell you about the latent space structure and the role of training data
   quantity/diversity?

In [ ]:
# Exercise B — Part 1: MNIST interpolation (train FC-GAN + interpolate)

In [ ]:
# Exercise B — Part 2: CelebA interpolation (using WGAN-GP from Exercise A)

**Exercise B — Written Analysis:**

*Compare the MNIST and CelebA interpolations. What differences do you observe? Why?*

---
# Exercise C — Hyperparameter Sensitivity: gamma (MNIST, low GPU)

**⚠️ You MUST use MNIST for this exercise** (not CelebA). This is to conserve GPU time.

**Requirements:**
1. Implement a WGAN-GP on MNIST (you can use a fully-connected or small CNN architecture).
   You can use `hp.Wasserstein_loss_fcn` and `hp.penalty_fcn` for the loss and gradient
   penalty, just as in LESSON 3B. Note: `hp.penalty_fcn` works correctly as long as the
   critic receives images in their original shape (e.g., `(batch, 1, 28, 28)`), even if
   the critic flattens them internally.
2. Test **3 gradient penalty gamma values: {1, 10, 50}**.
3. Train **15 epochs** for each gamma value.
4. For each gamma: show loss curves, generated samples, and diversity score
   (`hp.diversity_score` — higher values indicate more diverse outputs; low values suggest
   mode collapse).
5. **Written explanation** of which gamma works best and why.

In [ ]:
# Exercise C — WGAN-GP on MNIST with different gamma values

**Exercise C — Written Explanation:**

*Write your explanation here about which gamma works best and why.*

---
# Exercise D — Mode Collapse + Diversity Metric (MNIST, low GPU)

**⚠️ You MUST use MNIST for this exercise** (not CelebA). This is to conserve GPU time.

**Requirements:**
1. Train a GAN with settings that **deliberately cause mode collapse**.
   You must cause collapse through architectural or training imbalance — for example,
   an overpowered discriminator (much larger than the generator) or an unbalanced
   training ratio (e.g., training the discriminator many more times per generator step).
   **Do not use `z_dim=2`**, since that approach was already demonstrated in LESSON 3A.
   You must justify your choice.
   Train ~20 epochs.
2. Train a **healthy GAN** on MNIST with good settings. Train ~20 epochs.
3. Track `hp.diversity_score()` every epoch for **both** trainings.
   **Important:** `hp.train()` does not compute diversity scores internally. You will need
   to write your own epoch loop for **both** trainings (similar to the LESSON 3A FC-GAN
   loop) to track this metric per epoch.
4. Plot diversity score over epochs for both on the same figure.
5. Show generated images from both (collapsed vs healthy).
6. **Written explanation** of your findings and what the diversity metric reveals.

In [ ]:
# Exercise D — Mode collapse training (deliberately bad settings)

In [ ]:
# Exercise D — Healthy GAN training

In [ ]:
# Exercise D — Plot diversity scores and generated images

**Exercise D — Written Explanation:**

*Write your explanation here about mode collapse and the diversity metric.*

---
# Exercise E — Critical Thinking (written, no GPU)

Answer the following questions in markdown. Each answer should be substantive (50-150
words) and demonstrate understanding of the underlying concepts.

1. Why is generator loss alone a **bad indicator of image quality** for GANs?

2. Propose an **alternative evaluation criterion** for assessing generated image quality
   and explain why it's better.

3. How would you **detect mode collapse from loss curves alone** (without looking at
   generated images)?

**Exercise E — Answers:**

**1.** *Your answer here.*

**2.** *Your answer here.*

**3.** *Your answer here.*